# 7.1 Attention 概念入門

這份 Notebook 用小型向量範例說明 Query、Key、Value、attention score 與 attention weight。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立 token embeddings

假設句子有 4 個 token，每個 token 用 3 維向量表示。


In [ ]:
tokens = ['customer', 'refund', 'payment', 'delay']
embeddings = tf.constant([
    [1.0, 0.1, 0.0],
    [0.9, 0.2, 0.1],
    [0.8, 0.1, 0.2],
    [0.0, 1.0, 0.8],
], dtype=tf.float32)

pd.DataFrame(embeddings.numpy(), index=tokens, columns=['dim1', 'dim2', 'dim3'])


## 3. 用其中一個 token 當 Query

這裡用 `refund` 當 Query，觀察它和所有 Key 的相似度。


In [ ]:
query = embeddings[1:2]
keys = embeddings
values = embeddings

d_k = tf.cast(tf.shape(keys)[-1], tf.float32)
scores = tf.matmul(query, keys, transpose_b=True) / tf.sqrt(d_k)
weights = tf.nn.softmax(scores, axis=-1)
context = tf.matmul(weights, values)

print('scores:', scores.numpy().round(3))
print('weights:', weights.numpy().round(3))
print('context:', context.numpy().round(3))


## 4. 視覺化 attention weights


In [ ]:
plt.figure(figsize=(6, 2.2))
plt.bar(tokens, weights.numpy().ravel())
plt.title('Attention weights for query = refund')
plt.ylim(0, 1)
plt.show()


## 5. 對所有 token 計算 attention matrix


In [ ]:
all_scores = tf.matmul(embeddings, embeddings, transpose_b=True) / tf.sqrt(d_k)
all_weights = tf.nn.softmax(all_scores, axis=-1)

plt.figure(figsize=(5, 4))
plt.imshow(all_weights.numpy(), cmap='Blues')
plt.xticks(range(len(tokens)), tokens, rotation=30)
plt.yticks(range(len(tokens)), tokens)
plt.colorbar(label='attention weight')
plt.title('Attention weight matrix')
plt.tight_layout()
plt.show()

pd.DataFrame(all_weights.numpy(), index=tokens, columns=tokens).round(3)


## 6. 小結

Attention 會根據 Query 和 Key 的相似度產生權重，再用權重加總 Value。Transformer 裡的 self-attention 只是把這個流程擴展到整個序列與多個 attention head。
